In [ ]:
# start coding here
import numpy as np
import pandas as pd
from scipy import signal

def gillespie_two_state(k_forward, k_backward, initial_state=0, max_time=10.0, seed=None):
    """
    Gillespie algorithm for a two-state stochastic process.
    
    Parameters:
    - k_forward: rate constant for transition 0 -> 1
    - k_backward: rate constant for transition 1 -> 0
    - initial_state: starting state (0 or 1)
    - max_time: simulation end time
    - seed: random seed for reproducibility
    
    Returns:
    - times: array of transition times
    - states: array of states at each transition
    """
    if seed is not None:
        np.random.seed(seed)
    
    times = [0.0]
    states = [initial_state]
    current_state = initial_state
    current_time = 0.0
    
    while current_time < max_time:
        # Determine propensity functions based on current state
        if current_state == 0:
            propensity_forward = k_forward
            propensity_backward = 0
        else:
            propensity_forward = 0
            propensity_backward = k_backward
        
        total_propensity = propensity_forward + propensity_backward
        
        # Sample time to next event (exponential distribution)
        tau = np.random.exponential(1.0 / total_propensity)
        current_time += tau
        
        if current_time >= max_time:
            break
        
        # Determine which transition occurs
        if propensity_forward > 0:
            current_state = 1
        else:
            current_state = 0
        
        times.append(current_time)
        states.append(current_state)
    
    return np.array(times), np.array(states)




In [ ]:
# Parameters
k_on = 0.5    # rate 0 -> 1
k_off = 0.5   # rate 1 -> 0

# Run simulation
times_stoch, states_stoch = gillespie_two_state(k_on, k_off, initial_state=0, max_time=50.0, seed=42)
states_det = 0.5 * signal.square(2 * np.pi /10 * times_stoch ) + 0.5
df = pd.DataFrame({'time': times_stoch,
                   'states_stoch': states_stoch,
                   'states_det': states_det,})


In [ ]:
df.to_csv('two_state_gillespie_simulation.csv', sep='\t')